In [1]:
import pandas as pd
import re
import unicodedata

In [2]:
# Importamos el csv con la ruta relativa del repositorio 
food = pd.read_csv('../data/raw/openfoodfacts_nutrition_final_2025-12-10.csv')

In [3]:
food.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9804 entries, 0 to 9803
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                9804 non-null   int64  
 1   product_name        9566 non-null   object 
 2   brands              9430 non-null   object 
 3   countries           9802 non-null   object 
 4   quantity            8816 non-null   object 
 5   categories          9683 non-null   object 
 6   labels              7552 non-null   object 
 7   nutriscore_grade    9801 non-null   object 
 8   ecoscore_grade      9802 non-null   object 
 9   nova_group          8959 non-null   float64
 10  ingredients_text    9406 non-null   object 
 11  energy-kcal_100g    9280 non-null   float64
 12  fat_100g            9309 non-null   float64
 13  saturated-fat_100g  9140 non-null   float64
 14  carbohydrates_100g  9284 non-null   float64
 15  sugars_100g         9168 non-null   float64
 16  fiber_

## Unificación de países y selección del país principal 

In [4]:
# Hacemos una lista de todos los países y las formas en las que aparecen para hacer un diccionario que unifique todo
food["countries"].unique().tolist()

['Morocco',
 'Morocco,United States',
 'المغرب',
 "Belgium,Côte d'Ivoire,France,Germany,Guadeloupe,Italy,Luxembourg,Mali,Martinique,New Caledonia,Switzerland,United Kingdom",
 'Democratic Republic of the Congo,Mauritania,Morocco,United States',
 'Maroc,en:morocco',
 'Maroc',
 'France,Germany,Ireland,Italy,Netherlands,Spain,United Kingdom,United States',
 'Algeria,Belgium,France,French Polynesia,Germany,Guadeloupe,Hungary,Luxembourg,Martinique,Morocco,New Caledonia,Réunion,Spain,Switzerland,United States',
 'Algérie,Autriche,Belgique,Bulgarie,Canada,République tchèque,Finlande,France,Polynésie française,Allemagne,Irlande,Italie,Maurice,Maroc,Pays-Bas,Norvège,La Réunion,Roumanie,Singapour,Espagne,Suède,Suisse,Tunisie,Royaume-Uni',
 'Algérie,Belgique,Bulgarie,France,Allemagne,Guadeloupe,Inde,Italie,Luxembourg,Martinique,Maroc,La Réunion,Espagne,Suisse,Tunisie,Royaume-Uni,États-Unis',
 'Marocco',
 'Maroc,États-Unis',
 'Belgium, Bulgaria, France, en:switzerland',
 'Ireland,United Kingdom,Un

In [5]:
def filtrar_potencias(df, columna='countries'):
    
    """
    Filtra y normaliza la distribución geográfica del dataset.
    Se ha utilizado una lista exhaustiva de variantes lingüísticas y códigos ISO para alimentar 
    el diccionario de unificación, permitiendo el cálculo de la variable 'main_country' 
    para segmentar el análisis por mercados clave.
    """
    
    # 1. Definimos los patrones de búsqueda (Regex)
    patrones = {
        'Spain': r'Spain|España|Espagne|Espanha|ES|en:es',
        'France': r'France|Francia|França|FR|en:fr',
        'Germany': r'Germany|Alemania|Allemagne|Deutschland|DE|en:de',
        'United Kingdom': r'United Kingdom|UK|Reino Unido|Royaume-Uni|GB|en:gb|Great Britain',
        'Maghreb': r'Morocco|Maroc|Marruecos|Algeria|Algérie|Tunisia|Tunisie|MA|DZ|TN|en:ma|المغرب',
        'Italy': r'Italy|Italia|Italie|IT|en:it',
        'Benelux': r'Belgium|Belgique|België|Netherlands|Nederland|Pays-Bas|BE|NL',
        'Alpine (CH/AT)': r'Switzerland|Suisse|Schweiz|Austria|Österreich|CH|AT',
        'Nordics': r'Norway|Sweden|Sverige|Denmark|Danmark|Finland|Suomi|NO|SE|DK|FI',
        'Eastern Europe': r'Poland|Polska|Romania|România|Bulgaria|Czech|Hungary|PL|RO',
        'Americas': r'United States|USA|EEUU|Argentina|Mexico|México|Canada|US|MX|AR|CA',
        'Ireland': r'Ireland|Irlanda|en:ie|IE',
        'Portugal': r'Portugal|Portugália|en:pt|PT'
    }

    patron_global = '|'.join(patrones.values())
    
    # Filtramos el DataFrame original
    df_result = df[df[columna].str.contains(patron_global, case=False, na=False)].copy()

    # 2. Lógica de etiquetado priorizada
    def etiquetar_pais(row):
        val = str(row).lower()
        if any(term in val for term in ['spain', 'españa', 'espagne', 'es:']): return 'Spain'
        if any(term in val for term in ['france', 'frança', 'fr:']): return 'France'
        if any(term in val for term in ['germany', 'alemania', 'deutschland', 'de:']): return 'Germany'
        if any(term in val for term in ['united kingdom', 'uk', 'gb:', 'en:gb']): return 'United Kingdom'
        if any(term in val for term in ['morocco', 'maroc', 'marruecos', 'algeria', 'tunisia', 'ma:', 'المغرب']): return 'Maghreb'
        if any(term in val for term in ['switzerland', 'suisse', 'schweiz', 'austria', 'österreich', 'ch:', 'at:']): return 'Alpine (CH/AT)'
        if any(term in val for term in ['sweden', 'denmark', 'finland', 'norway', 'norge']): return 'Nordics'
        if any(term in val for term in ['belgium', 'belgique', 'netherlands', 'holland']): return 'Benelux'
        if any(term in val for term in ['poland', 'romania', 'bulgaria', 'hungary', 'czech']): return 'Eastern Europe'
        if any(term in val for term in ['ireland', 'ie:']): return 'Ireland'
        if any(term in val for term in ['italy', 'italia', 'it:']): return 'Italy'
        if any(term in val for term in ['usa', 'united states', 'mexico', 'argentina', 'canada']): return 'Americas'
        if any(term in val for term in ['portugal', 'pt:']): return 'Portugal'
        return 'Other / Global'

    df_result['main_country'] = df_result[columna].apply(etiquetar_pais)
    
    # 3. Métricas de Cobertura
    total_original = len(df)
    total_clasificado = len(df_result)
    porcentaje_cobertura = (total_clasificado / total_original) * 100
    
    counts = df_result['main_country'].value_counts()
    percentages = (counts / total_clasificado) * 100
    
    tabla_final = pd.DataFrame({
        'Número de Productos': counts,
        'Porcentaje (%)': percentages.round(2)
    })
    
    print("\n" + "="*50)
    print("      INFORME DE DISTRIBUCIÓN GEOGRÁFICA")
    print("="*50)
    print(tabla_final)
    print("-" * 50)
    print(f"RESUMEN: Se ha clasificado el {porcentaje_cobertura:.2f}% del dataset original ({total_clasificado} de {total_original} productos).")
    print("="*50)
    
    return df_result

# Ejecución
food_countries = filtrar_potencias(food)


      INFORME DE DISTRIBUCIÓN GEOGRÁFICA
                Número de Productos  Porcentaje (%)
main_country                                       
France                         4836           49.66
United Kingdom                 1471           15.11
Spain                          1119           11.49
Other / Global                  836            8.58
Maghreb                         685            7.03
Germany                         448            4.60
Americas                         80            0.82
Portugal                         63            0.65
Eastern Europe                   52            0.53
Italy                            51            0.52
Ireland                          35            0.36
Alpine (CH/AT)                   30            0.31
Benelux                          20            0.21
Nordics                          12            0.12
--------------------------------------------------
RESUMEN: Se ha clasificado el 99.33% del dataset original (9738 de 9804 pro

NOTA: Para maximizar la precisión, se desglosó repetidamente la categoría 'Other', reintegrando nuevos términos en el diccionario de unificación. Este enfoque permitió reclasificar productos residuales hasta cubrir el 99.33% del dataset.

## Clasificación de productos por categoría

In [6]:
def unificacion_categorias(df, columna='categories'):
    
    """
    Realiza una normalización semántica y lingüística de las categorías de productos.
    El proceso utiliza un motor de reglas basado en palabras clave (keywords) para 
    mapear miles de etiquetas originales en 13 categorías estratégicas. 
    Se ha aplicado una jerarquía de prioridades para resolver ambigüedades, 
    especialmente en productos plant-based y de conveniencia.
    """
    
    def limpiar(txt):
        if pd.isna(txt): return "Otros"
        txt = re.sub(r'[a-z]{2}:', '', str(txt)).lower()
        
        # 1. BEBIDAS VEGETALES Y SUSTITUTOS (Prioridad Máxima)
        if any(p in txt for p in ['vegetarian nuggets', 'fleisch-ersatzprodukte', 'soy-based-drinks', 'vegetarian patties', 'vegetarian sausages', 'soy-based', 'crèmes végétales', 'vege mince', 'mince', 'soy-based drinks', 'galettes de soja', 'vegan', 'oat based drinks', 'galettes de soja', 'soja', 'vegetable-based', 'galettes végétariennes', 'produits sans gluten', 'plantaardige', 'haverdranken', 'melkvervangers', 'galettes végétariennes', 'milk substitute', 'substitut de lait', 'substitute', 'alternative', 'vegetal milk', 'leche vegetal', 'lait végétal', 'tofu', 'seitan', 'heura', 'vegan meat']):
            return 'plant_based_alternatives'

        # 2. NUTRICIÓN INFANTIL (Tienen un Nutri-Score muy específico)
        if any(p in txt for p in ['cerelac', 'baby', 'bébé', 'infantil', 'bebé', 'potitos', 'babyfood', 'toddler', 'children pate']):
            return 'baby_food'

        # 3. PLATOS PREPARADOS Y CONVENIENCIA
        if any(p in txt for p in ['surgelés', 'pommes noisettes', 'cod acras', 'feuilles de brick', 'caldos', 'caldos líquidos', 'brou', 'spinach bites', 'vegetable rice', 'vegetable patties', 'tortilla', 'comidas-preparadas', 'courgettes cuisinées', 'prepared vegetables', 'puré', 'black dhal', 'kimchi', 'dumplings', 'purées', 'puree', 'luncheon', 'اللحوم المصنعة', 'refeições', 'refeições para micro-ondas', 'processed vegetables', 'prepared vegetables', 'maaltijden', 'ready-made', 'comidas-preparadas', 'tortilla', 'tortillas-de-patata', 'taboule', 'maaltijden', 'instant-noodles', 'tabouleh', 'taboulehs', 'kant-en-klaar maaltijden', 'maaltijden', 'hash browns', 'fries', 'chips and fries', 'gravy', 'stuffing', 'soups', 'bouillon', 'broths', 'meals', 'ready-to-eat', 'soups', 'bouillon', 'broths', 'stuffing', 'fries', 'potatoes', 'meals', 'ready', 'préparé', 'preparado', 'pizza', 'lasagna', 'sandwich', 'hummus', 'sushi', 'take away']):
            return 'ready_to_eat'

        # 4. LÁCTEOS Y HUEVOS (Animal)
        if any(p in txt for p in ['mosarella', 'ferments-lactiques', 'zuivelproducten', 'mliječni proizvodi', 'slastice', 'raïb', 'tejtermékek', 'krémek', 'كعكة', 'crèmes fraîches', 'creme', 'masła', 'nabiał', 'tłuszcze mleczne', 'slastice', 'pjenice', 'mliječni', 'yaourt', 'ياغورت', 'laticínios', 'natilhas', 'chees', 'zuivelproducten', 'cream', 'kefir', 'skir', 'млечни', 'latte', 'milchprodukte', 'creams', 'oeufs', 'œufs', 'oeufs', 'œufs', 'creams', 'dairy', 'laitier', 'lácteo', 'cheese', 'fromage', 'queso', 'yogurt', 'yaourt', 'milk', 'lait', 'leche', 'egg', 'oeuf', 'huevo']):
            if 'vegan' in txt or 'plant-based' in txt: return 'plant_based_alternatives'
            return 'dairy_eggs'

        # 5. CARNE Y PESCADO (Animal)
        if any(p in txt for p in ['tonijn', 'conserven', 'tonijn op olie', 'turkey bacon', 'tuna-in-catalan', 'tuna', 'bacon', 'canned-tunas', 'mackerel-fillets', 'tunas', 'mackerel', 'chipolatas', 'charcuterie', 'charcuteries', 'charcuterie de volaille', 'vissen', 'makreel', 'pechuga de pavo', 'meat', 'viande', 'carne', 'fish', 'poisson', 'pescado', 'atún', 'thon', 'sardine', 'chicken', 'poulet', 'pollo', 'ham', 'jambon', 'jamón']):
            if 'vegan' in txt or 'plant-based' in txt: return 'plant_based_alternatives'
            return 'meat_fish'

        # 6. DULCES, SNACKS Y POSTRES
        if any(p in txt for p in ['kakao', 'kakaov', 'zrna kakaa', 'pommes noisettes', 'gum', 'pastry', 'croissants', 'waffles', 'caramel', 'matzos', 'cracker', 'waffles', 'caramel', 'chocholat', 'malt loaf', 'cakes', 'закуски', 'сладки закуски', 'przekąski', 'słodkie przekąski', 'batoniki', 'cake', 'cocoa cake','cakes', 'pufuleți', 'biscuit', 'بسكويت', 'barre céréalière', 'confiture', 'confitures', 'biscuit', 'bonbons', 'aliments de pâques', 'lentil & pea cakes', 'appetizers', 'grickalice', 'slatki zalogaji', 'biscoitos', 'bolos', 'bonbons', 'biscuit', 'lanches doces', 'gelatina', 'biscotti', 'wafers', 'chewing-gum', 'flan', 'crisps', 'crackers', 'confectioneries', 'bars', 'chips', 'crackers', 'wafers', 'confectioneries', 'flans', 'bars', 'chips', 'crisps', 'snack', 'biscuits', 'gateaux', 'galletas', 'chocolat', 'candy', 'confectionery', 'sweet', 'sucré', 'dessert', 'ice cream', 'glace', 'helado']):
            return 'snacks_sweets'

        # 7. GRASAS, ACEITES Y SALSAS
        if any(p in txt for p in ['dips', 'vetten', 'pate', 'huil', 'sriracha', 'sauces', 'ouile', 'matière grasse', 'matières grasses', 'sojasauzen', 'chutneys', 'aceto balsamico', 'essige', 'smaakmakers', 'sauzen', 'olivenöle', 'butter', 'beurre', 'moutarde', 'mustard', 'senf', 'jams', 'confitures', 'beurres de cacahuètes', 'syrups', 'sirops', 'agave', 'confitures', 'butter', 'erdnussbutter', 'oil', 'huile', 'aceite', 'sauce', 'condiment', 'ketchup', 'pesto', 'spread', 'tartiner', 'untable', 'mayo', 'margarine', 'vinegar', 'vinaigre', 'vinagre']):
            return 'fats_sauces'

        # 8. CEREALES, PAN Y PASTAS
        if any(p in txt for p in ['wheats', 'rolled oats', 'ciabatta', 'oats', 'nourriture', 'malt loaf', 'spaghetti', 'extruded flakes', 'flakes', 'céréalière', 'germes de blé', 'granola', 'porridge', 'sourdough', 'brioches', 'germé de blé', 'pahuljice', 'crêpes', 'galettes de froment', 'tartines craquantes', 'pahuljice', 'zobene pahuljice', 'germe de blé', 'knäckebrote', 'biscottes', 'semoule', 'rusks', 'müsli', 'getreide', 'crumpets', 'rices', 'frühstückscerealien', 'flour', 'farine', 'knäckebrote', 'crumpets', 'rices', 'müslis', 'getreide', 'frühstückscerealien', 'céréale', 'cereals', 'bread', 'pain', 'pan', 'pão', 'pasta', 'pâte', 'muesli', 'breakfast', 'flour', 'farine', 'harina']):
            return 'cereals_bread'

        # 9. BEBIDAS (Aguas, Refrescos, Café, Té)
        txt_sin_ruido = txt.replace('plant-based foods and beverages', '')
        if any(p in txt_sin_ruido for p in ['entergy', 'dranken en drankbereidingen', 'frisdranken', 'tonics', 'gesuikerde dranken', 'напитки', 'вода', 'limonaden', 'gezuckerte getränke', 'juomat', 'tisanes', 'infusions', 'apa', 'napoje', 'woda', 'gazirana', 'sportdrank', 'chicorée soluble', 'chicorée', 'sprite', 'مشروب', 'getränkezubereitungen', 'energy-drinks', 'pića', 'pripravci za piće', 'soft drinks', 'kola kola', 'gazirana pića', 'boissons', 'beverage', 'boisson', 'bebida', 'bevande', 'water', 'eau', 'agua', 'juice', 'jus', 'zumo', 'soda', 'cola', 'coffee', 'café', 'tea', 'thé', 'té']):
            return 'beverage'

        # 10. ALCOHOL (Importante: Nutri-Score no suele aplicar aquí)
        if any(p in txt for p in ['alcohol', 'vin', 'wine', 'beer', 'bière', 'cerveza', 'spirit', 'liqueur']):
            return 'alcoholic_beverages'

        # 11. SUPLEMENTOS Y NUTRICIÓN DEPORTIVA
        if any(p in txt for p in ['supplement', 'complément', 'suplemento', 'protein powder', 'whey', 'vitamin', 'vitamine', 'nahrungsergänzungsmittel', 'proteinpulver', 'proteinshakes', 'bodybuilder']):
            return 'supplements_nutrition'

        # 12. VEGETALES, FRUTAS Y LEGUMBRES (Frescos/Conservas simples)
        if any(p in txt for p in ['castañas', 'red kidney beans', 'kidney beans', 'frozzen vegetables', 'frozen-vegetables', 'kasvipohjaiset', 'hedelmä', 'seeds', 'canned lentils', 'lentils', 'courgettes', 'prunes', 'dried-prunes', 'semillas', 'chía', 'maïs frit', 'corn', 'walnüsse', 'seeds', 'cauliflower rice', 'frozen-vegetables', 'nut', 'noix', 'vegetal', 'plant-based', 'fruit', 'fruta', 'légume', 'verdura', 'tomato', 'tomate', 'nut', 'noix', 'frutos secos', 'legume', 'legumbre']):
            return 'plant_based'
            
        # 13. ESPECIAS Y POLVOS PARA COCINAR
        if any(p in txt for p in ['vanilla', 'extract', 'cane-sugar', 'sucre-de-canne', 'endulzantes', 'ciclamato', 'sugars', 'ingredients', 'المكونات', 'aides culinaires', 'honey', 'miels', 'honige', 'bienenprodukte', 'cannelle', 'aides culinaires', 'edulcoranti', 'sciroppi', 'acero', 'siroppi', 'sucres roux', 'cassonades', 'yeast', 'sucres', 'cassonades', 'agave', 'sirops', 'additifs', 'édulcorants', 'sucralose', 'sweetener', 'spice', 'épice', 'especia', 'baking', 'levure', 'powder', 'sel', 'salt', 'sal']):
            return 'spices_baking_sugars'

        return "Otros"

    df['category_unified'] = df[columna].apply(limpiar)
    
    # Reporte de Cobertura
    total = len(df)
    otros_count = df[df['category_unified'] == 'Otros'].shape[0]
    print(f"Limpieza completa. 'Otros' se ha reducido al {(otros_count/total)*100:.2f}% del dataset.")
    
    return df

food_countries = unificacion_categorias(food_countries)
print(food_countries['category_unified'].value_counts())

Limpieza completa. 'Otros' se ha reducido al 1.43% del dataset.
category_unified
snacks_sweets               2593
ready_to_eat                1896
dairy_eggs                  1674
fats_sauces                  945
beverage                     774
cereals_bread                589
plant_based_alternatives     525
meat_fish                    349
plant_based                  145
Otros                        139
spices_baking_sugars          71
supplements_nutrition         18
baby_food                     15
alcoholic_beverages            5
Name: count, dtype: int64


NOTA: De la misma forma que se procedió con la normalización de países, se aplicó un proceso de limpieza profunda para las categorías. Para maximizar la precisión, se desglosó repetidamente la categoría 'Other', reintegrando nuevos términos detectados en el diccionario de unificación. Este enfoque de refinamiento iterativo permitió reclasificar productos residuales hasta cubrir el 98.57% del dataset.

### Resumen de las 13 Categorías Resultantes

Plant-based Alts: Sustitutos tecnológicos.

Baby Food: Nutrición específica infantil.

Ready-to-eat: Conveniencia y platos preparados.

Dairy/Eggs: Origen animal.

Meat/Fish: Origen animal.

Snacks/Sweets: Ultraprocesados placenteros.

Fats/Sauces: Aceites y condimentos líquidos.

Cereals/Bread: Carbohidratos base.

Beverage: Bebidas no alcohólicas.

Alcohol: Bebidas fermentadas/destiladas.

Supplements: Nutrición deportiva/salud.

Plant-based: Vegetales y frutas "en bruto" o poco procesados.

Spices/Baking: Ingredientes de cocina.

## Clasificación de productos por marca

In [7]:
def normalizar(texto):
    
    """
MOTOR DE NORMALIZACIÓN Y CLASIFICACIÓN DE CORPORACIONES (Mapping Jerárquico)

Objetivo: 
Resolver la fragmentación de marcas en OpenFoodFacts, agrupando miles de variantes 
(ej: 'Nestlé', 'Nescafé', 'Maggi') bajo su entidad matriz (ej: 'Nestlé') para 
permitir un análisis de mercado por grupos corporativos.

Metodología en 3 etapas:
1. Normalización Fonética y Tipográfica: 
   La función 'normalizar' utiliza el estándar Unicode (NFD) para eliminar acentos, 
   cedillas y caracteres especiales. Transforma 'Nestlé' en 'nestle', asegurando 
   que errores de escritura en el dataset no afecten al agrupamiento.

2. Diccionario de Mapeo Multilevel:
   Estructura de datos exhaustiva que clasifica marcas por sectores estratégicos:
   - Grandes Multinacionales (Big Food).
   - Marcas de Distribuidor (Retailers/MDD): Hacendado, Lidl, Carrefour, etc.
   - Grupos Especializados (Lácteos, Bio, Bebidas).

3. Algoritmo de Búsqueda Optimizado (Motor de Búsqueda):
   - Prioridad por Longitud: Se ordenan las marcas de mayor a menor longitud para 
     evitar falsos positivos (ej: no confundir 'Mars' con 'Marshmallow').
   - Regex con 'Word Boundaries' (\b): Garantiza que se encuentre la marca exacta 
     y no una cadena de texto dentro de otra palabra.
   - Memoización (Memo): Sistema de caché para acelerar el procesamiento, calculando 
     solo una vez cada variante única de marca.
"""
    
    if pd.isna(texto): return ""
    # Transforma 'Nestlé' en 'Nestle' y pasa a minúsculas
    texto = unicode_str = unicodedata.normalize('NFD', str(texto))
    return "".join([c for c in unicode_str if unicodedata.category(c) != 'Mn']).lower().strip()

mapping = {
    # ======================
    # GRANDES MULTINACIONALES
    # ======================
    'Nestlé': [
        'nestle', 'nescafe', 'nescafé', 'nesquik', 'maggi', 'chocapic',
        'garden gourmet', 'le chocolat', 'lion', 'kitkat', 'after eight',
        'cookie crisp', 'golden grahams', 'herta', 'la laitiere', 'la laitière',
        'la laitiere, la laitière', 'mousline', 'vittel', 'perrier', 'badoit',
        'solis', 'buino', 'ricore', 'dolce gusto', 'cailler', 'ovomaltine',
        'ovo maltine', 'cappuccino, gold cappuccino, nescafé'
    ],

    'Unilever': [
        'unilever', 'marmite', 'hellmann', "hellman's", 'hellmanns',
        'knorr', 'knoor', 'amora', 'maizena', 'ben & jerry', 'ben & jerry\'s',
        'maille', 'frigo', 'ligeresa', 'calve', 'calvé', 'flora', 'planta fin',
        'flora proactiv', "fruit d'or", "fruit d'or proactiv, pro activ, proactiv, proactiv expert",
        "fruit d'or,becel", "fruit d'or proactiv,proactiv,proactiv expert,pro activ",
        'lipton', 'benedicta', 'bénédicta', 'skip', 'dove',
        'pukka', 'fruittare', 'the vegetarian butcher', 'magnum', 'magnum', 'miko', 'graze'
    ],

    'Mondelez': [
        'mondelez', 'lu', 'lu ', ' lu', 'lu,', 'lu.', 'milka', 'oreo',
        'toblerone', 'tuc', 'belvita', 'cadbury', 'philadelphia',
        'cote d’or', "cote d'or", "côte d'or", 'granola', 'heudebert',
        'pelletier', 'mikado', 'fontaneda', 'principe', 'príncipe', 'barny',
        'oscar mayer', 'belin', 'royal', 'daunat', 'jacob', 'mcvitie', 'nippon'
    ],

    'Danone': [
        'danone', 'activia', 'evian', 'volvic', 'alpro', 'hipro',
        'savane', 'font vella', 'lanjarón', 'les 2 vaches', 'oatly',
        'la salvetat', 'salvetat', 'vrai', 'charles & alice', 'michel & augustin',
        'michel et augustin', 'siggi', 'nyakas', 'danao'
    ],

    'Ferrero': [
        'ferrero', 'nutella', 'kinder', 'rocher', 'tic tac', 'tictac',
        'raffaello', 'mon cheri', 'mon chéri', 'mon chéri', "MON CHÉRI"
    ],

    'PepsiCo': [
        'pepsico', 'pepsi', 'lays', "lay's", 'lay&#039;s', 'doritos', 'cheetos',
        'alvalle', 'quaker', 'gatorade', 'tropicana', 'cruesli', 'matutano',
        'benenuts', 'sun bites', 'sun bites', 'walkers', 'AQUAFINA'
    ],

    'Coca-Cola': [
        'coca-cola', 'coca cola', 'coke', 'fanta', 'sprite', 'fuze tea',
        'aquarius', 'minute maid', 'pulco', 'honest', 'innocent', 'powerade',
        'monster energy', 'nestea', 'adez', 'cappy', 'smartwater',
        'the coca cola company'
    ],

    'Kellogg': [
        'kellogg', "kellog's", 'special k', 'all-bran', 'frosties',
        'smacks', 'tresor', 'cheerios', 'cheerios multigrain', 'pringles', 'Pringles', 'PRINGLES'
    ],

    'General Mills': [
        'general mills', 'nature valley', 'old el paso', 'haagen-dazs',
        'häagen-dazs', 'jus rol', 'pillsbury', 'green giant', 'gigante verde', 'yoplait', 'yop', 'petit filous'
    ],

    'Kraft Heinz': [
        'heinz', 'kraft', 'orlando', 'hp sauce', 'lea & perrins'
    ],

    'Mars Wrigley': [
        'mars', 'm&m', "m&m's", 'snickers', 'twix', 'orbit', 'skittles',
        'maltesers', 'celebrations', 'reese', 'whiskas', 'pedigree',
        'ebly', "ben's original", 'suzi wan'
    ],

    'Dr. Oetker': [
        'dr. oetker', 'dr oetker', 'alsa', 'ancel', 'condi', 'muesli crunchy',
        'muesli crunchy'
    ],

    'Deoleo / Aceites': [
        'deoleo', 'carapelli', 'bertolli', 'carbonell', 'koipe', 'hula'
    ],

    'Valeo Foods': [
        'valeo foods', 'balconi', 'kettle', 'kettle chips', 'rowan hill'
    ],

    'Princes Group': [
        'princes', 'napolina', 'juver'
    ],

    'Ecotone / Wessanen': [
        'ecotone', 'wessanen', 'bjorg', 'alter eco',
        'allos', 'clipper', 'tanoshi', 'zonnatura', 'whole earth', 'isitar'
    ],

    'Upfield': [
        'becel', "fruit d'or", 'proactiv', 'pro activ',
        "fruit d'or proactiv,proactiv,proactiv expert,pro activ",
        "fruit d'or,becel", "fruit d'or proactiv, pro activ, proactiv, proactiv expert",
        "fruit d'or proactiv,proactiv,proactiv expert, pro activ"
    ],

    # ======================
    # MARCAS DE DISTRIBUIDOR / RETAILERS
    # ======================

    'Hacendado / Mercadona': [
        'hacendado', 'mercadona', 'deliplus', 'compy', 'bosque verde',
        'entrepinares', 'facendado', 'FACENDADO'
    ],

    'Lidl': [
        'lidl', 'alesto', 'milbona', 'fin carré', 'fin carre', 'sondey',
        'freshona', 'crownfield', 'vemondo', 'vermondo', 'baresa',
        'j.d. gross', 'bellarom', 'combino', 'italiamo', 'snack day',
        'envia', 'nixe', 'maribel', 'saguaro', 'trattoria', 'gelatelli',
        'toque du chef', 'duc de coeur', 'deluxe', "vitad'or",
        "vita d'or", 'belbake', 'primadonna', 'solesita', 'tastino',
        'harvest basket', 'maître jean pierre', 'maitre jean pierre',
        'mister choc', 'favourina', 'favorina', 'rivercote', 'solevita'
    ],

    'Carrefour / Système U / Intermarché': [
        'carrefour', 'reflets de france', 'eco +', 'eco+',
        'nos regions ont du talent', 'nos régions ont du talent',
        'nos regions ont du talent', 'nos régions ont du talent',
        'u bio', 'magasins u', 'u ', 'u,', 'u', 'paturages', 'pâturages',
        'páturages', 'paturages, sélection des mousquetaires',
        'sélection des mousquetaires', 'les mousquetaires',
        'nos regions ont du talent', 'nos régions ont du talent',
        'itineraires des saveurs', 'itinéraire des saveurs',
        'itinéraire des saveurs, les mousquetaires',
        'itineraires de nos regions',
        'saveurs de nos regions', 'saveurs de nos régions',
        'saveurs de nos régions', 'saveurs de nos regions'
    ],

    'Leclerc': [
        'e.leclerc', 'bio village', 'marque repère', 'marque repere',
        'saveurs de nos régions', 'saveurs de nos regions'
    ],
    
    'Aldi': [
        'aldi', 'harvest morn', 'savour bakes', 'specially selected', 
        'the foodie market', 'village bakery', 'bramwells', 'brooklea', 'four seasons', 'valencia'
    ],

    'Retail Internacional (Otros)': [
        'tesco', 'sainsbury', 'waitrose', 'marks & spencer', 'marks and spencer',
        'm&s', 'auchan', 'migros', 'coop', 'picard', 'costco',
        'asda', 'morrisons', 'système u', 'dia', 'el corte inglés',
        'mcennedy', 'stockwell', 'monoprix', 'woolworths food',
        'co op', 'the co-operative loved by us',
        'kirkland', 'kirkland signature', "member's mark"
    ],
    
    'Intermarche': [
        'intermarche', 'intermarché', 'chabrior', "bouton d'or"
    ],

    # ======================
    # GRANDES GRUPOS LÁCTEOS / QUESOS
    # ======================

    'Lactalis / Bel': [
        'lactalis', 'président', 'president', 'galbani', 'lactel', 'puleva',
        'lauki', 'société', 'société', 'bridelice', 'bridelight',
        'kiri', 'babybel', 'la vache qui rit', 'boursin', 'leerdammer',
        'candia', 'milsa', 'pilos', 'grand fermage', 'le petit basque',
        'la laitìère', 'la laitière', 'la laitiere', 'nurishh'
    ],

    'Savencia': [
        'savencia', 'st moret', 'saint moret', 'tartare', 'caprice des dieux',
        'saint agur', 'elle & vire', 'elle&vire', 'cœur de lion', 'chavroux',
        'bordeau chesnel', 'le rustique', 'fromarsac', 'rians'
    ],

    'FrieslandCampina': [
        'frieslandcampina', 'milko', 'longley farm'  # milko depende de país, se incluye como aproximación
    ],

    'Arla': [
        'arla', 'castello', 'lurpak'  # si lo tienes, añade otras marcas arla
    ],

    # ======================
    # OTROS GRANDES GRUPOS EUROPEOS
    # ======================

    'Barilla / Pasta': [
        'barilla', 'wasa', 'mulino bianco', 'harry', 'harrys',
        'de cecco', 'rummo', 'panzani', 'giovanni rana', 'rana',
        'misura', 'garofalo', 'jacquet', 'gran cereale', 'gran cereale'
    ],

    'Associated British Foods (ABF)': [
        'jordans', 'dorset cereals', 'twinings', 'ryvita', 'patak',
        'primark', 'allinson', 'kingsmill', 'silver spoon'
    ],

    'Ebro Foods / Mutti': [
        'ebro foods', 'brillante', 'sos', 'lustucru', 'taureau ailé',
        'tilda', 'mutti', 'liebig'
    ],
    
    'GB Foods': [
        'star', 'gallina blanca', 'avecrem', 'yatekomo'

    ],

    'Grandes Grupos Europeos': [
        'fleury michon', 'fleurs michon', 'bonne maman', 'andros',
        'gerblé', 'gerble', 'bonduelle', 'brioche pasquier', 'pasquier',
        'st michel', 'saint michel', 'la boulangère', 'la boulangere',
        'sodebo', 'tipiak', 'gullon', 'gullón', 'cuetara', 'cuétara',
        'mccain', 'mc cain', 'findus', 'bimbo', 'haribo', 'ricola',
        'st hubert', 'paysan breton', 'lotus', 'weetabix', 'pascual',
        'don simón', 'don simon', 'elpozo', 'campofrío', 'campofrio',
        'marie', 'petit navire', 'saupiquet', 'connetable', 'connétable',
        'poulain', 'lucien georgelin', 'pierre martinet', "d'aucy",
        'soignon', 'lesieur', 'quorn', 'entremont', 'william saurin',
        'jaouda', 'chergui', 'aicha', 'cassegrain', 'brets', "bret's",
        'warburtons', 'vico', 'seeberger', 'puget', 'primevère',
        'brossard', 'le gaulois', 'lutti', 'lune de miel', 'francine',
        'le ster', 'coraya', 'storck', 'hovis', 'tramier', 'yeo valley',
        'filippo berio', 'kikkoman', 'saclà', 'lindt', 'sprüngli',
        'bahlsen', 'bahlsen,leibniz', 'leibniz', 'Lotus Bakeries', 
        'lotus', 'nakd', 'bear', 'biscoff'

    ],

    # ======================
    # BIO / SALUD (GRUPOS GRANDES)
    # ======================

    'Léa Nature / Bio Salud': [
        'jardin bio', 'jardin bio étic', 'lea nature', 'léa nature',
        'karelea', 'bisson', 'evernat', 'ethiquable', 'éthiquable',
        'cereal bio', 'céréal bio', 'sojasun', 'taifun', 'soy',
        'happyvore', 'ecomil', 'eco mil', 'alnatura', 'huel',
        'deliciously ella', 'violife', 'pure via', 'dmbio', 'bio u',
        'isola bio', 'schar', 'kallo', 'bio & me', 'nairns', 'proper', 'crosta & mollica'
    ],

    # ======================
    # BEBIDAS / ALCOHOL
    # ======================

    'Bebidas no alcohólicas (otros grupos)': [
        'red bull', 'redbull', 'schweppes', 'orangina', 'teisseire',
        'cristaline', 'joker', 'pago', 'solan de cabras', 'bezoya',
        'san benedetto', 'oasis', 'robinsons', 'vita coco', 'champomy',
        'mogu mogu', 'mogu mogu (sappé)', 'thanon', 'campo largo',
        'hawai', 'HAWAI', 'champomy'
    ],

    'Cerveceras / Alcohol': [
        'estrella damm', 'estrella galicia', 'heineken', 'peroni',
        'budweiser', 'mahou', '1664', 'leffe', 'grimbergen',
        'tourtel', 'tourtel twist', 'tourtel twist citron',
        'tourtel twist framboise', 'tourtel twist pêche'
    ],

    # ======================
    # Especialistas / Líderes de Categoría
    # ======================

    'CHO Group (Terra Delyssa)': [
        'terra delyssa', 'terra delyssa estate', 'TERRA DELYSSA'
    ],

    'Ekibio / Le Pain des Fleurs (Léa Compagnie Biodiversité)': [
        'ekibio, le pain des fleurs', 'ekibio,le pain des fleurs',
        'le pain des fleurs', 'le pain des fleurs, ekibio',
        'ekibio', 'le moulin du pivert', 'le moulin du pivert', 'priméal'
    ],

    'La Fournée Dorée (grupo familiar francés)': [
        'la fournée dorée', 'la fournee doree', 'la fournée dorée',
        'la fournée dorée', 'la fournee doree'
    ],

    'Heura Foods': [
        'heura'
    ],

    'Samyang': [
        'samyang', 'buldak,samyang', 'buldak'
    ],

    'Lee Kum Kee': [
        'lee kum kee'
    ],

    'Ayam': [
        'ayam'
    ],

    'Artiach (Grupo Bimbo)': [
        'artiach'
    ],

    'Birds Eye / Nomad Foods': [
        'birds eye'
    ],

    'Mission (Gruma)': [
        'mission', 'gruma,mission wrap'
    ],

    'Mizkan': [
        'mizkan'
    ],

    'Mentos / Perfetti Van Melle': [
        'mentos', 'mentos,perfetti'
    ],

    'Tabasco / McIlhenny': [
        'tabasco', 'mcIlhenny company, tabasco', 'avery island la, mcihenny company, tabasco'
    ],

    'Rügenwalder Mühle': [
        'rügenwalder mühle'
    ],

    'Ritter Sport': [
        'ritter sport', 'RITTER SPORT'
    ],

    'Idilia Foods': [
        'valor', 'VALOR', 'nocilla', 'cola cao', 'colacao', 'colacaoidilia foods'
    ],
    
    'Anouar Invest': [
        'excelo', 'jibal', 'badaouia'
    ]
    
}

def clasificar_marcas(df, columna_brands, diccionario):
    # APLANAR Y NORMALIZAR EL DICCIONARIO
    flat_mapping = []
    for corp, marcas in diccionario.items():
        for m in marcas:
            # Guardamos la marca del diccionario ya normalizada
            flat_mapping.append((normalizar(m), corp))
    
    # Ordenar por longitud para máxima precisión
    flat_mapping.sort(key=lambda x: len(x[0]), reverse=True)

    memo = {}

    def motor_busqueda(texto_original):
        if pd.isna(texto_original) or texto_original == "": return 'Desconocido'
        
        # Normalizamos la marca que viene del dataset
        texto_norm = normalizar(texto_original)
        
        if texto_norm in memo: return memo[texto_norm]

        # Búsqueda optimizada
        for marca_hija_norm, corp_padre in flat_mapping:
            if marca_hija_norm in texto_norm:
                if re.search(rf'\b{re.escape(marca_hija_norm)}\b', texto_norm):
                    memo[texto_norm] = corp_padre
                    return corp_padre
                
        memo[texto_norm] = 'Otras / Local'
        return 'Otras / Local'

    # Ejecución sobre valores únicos
    marcas_unicas = df[columna_brands].unique()
    mapa_final = {marca: motor_busqueda(marca) for marca in marcas_unicas}
    
    df['corporation'] = df[columna_brands].map(mapa_final)
    return df

# --- EJECUCIÓN ---
food_countries = clasificar_marcas(food_countries, 'brands', mapping)

# --- RESULTADOS ---
cobertura = (food_countries['corporation'] != 'Otras / Local').mean() * 100
print(f"🚀 Nueva Cobertura tras Normalización: {cobertura:.2f}%")
print(food_countries['corporation'].value_counts().head(10))

🚀 Nueva Cobertura tras Normalización: 72.89%
corporation
Otras / Local                          2640
Grandes Grupos Europeos                1238
Lidl                                    682
Retail Internacional (Otros)            605
Desconocido                             374
Hacendado / Mercadona                   314
Carrefour / Système U / Intermarché     303
Nestlé                                  297
Mondelez                                269
Danone                                  265
Name: count, dtype: int64


NOTA: Se ha logrado una cobertura del 72.89% mediante un proceso de normalización avanzada y mapeo jerárquico. Al igual que en el análisis geográfico, se procedió de forma iterativa analizando los residuos de la categoría 'Otras / Local'. Este desglose permitió identificar nuevos patrones de marcas y retroalimentar el diccionario de unificación repetidamente, maximizando la clasificación y convirtiendo datos dispersos en información corporativa accionable.

## Clasificación de productos con etiqueta Bio/Eco/Organic

In [8]:
# 1. Definimos el patrón de búsqueda con todas las variantes internacionales
# Añadimos 'ab agriculture biologique' que es muy común en productos de Francia
bio_pattern = (
    r'(?i)\b(?:'
    # Términos generales y prefijos
    r'bio|eco|organic|organico|biologique|biologico|biologisch|ekologisk|ekologiczny|ökologisch|öko|'
    r'biomilch|bio-organic|bio-européen|bio-équitable|bio-dynamie|'
    
    # Francés (AB y variantes)
    r'agriculture biologique|ab agriculture biologique|nature et progrès|biopartenaire|'
    
    # Alemán y Centroeuropa
    r'bioland|naturland|demeter|eg-öko-verordnung|eu-öko-verordnung|'
    
    # Español / Italiano / Portugués
    r'ecológico|ecologico|biológico|biologico ue|orgânico|organico eu|'
    
    # Inglés y otros
    r'eu organic|usda organic|certified organic|aco certified organic|soil association organic|'
    r'organic food federation|canada organic|'
    
    # Sellos de certificación técnica
    r'[a-z]{2}-bio-\d+|[a-z]{2}-öko-\d+|[a-z]{2}-org-\d+|[a-z]{2}-eko-\d+|'
    r'at-bio-301|at-bio-901|be-bio-01|be-bio-02|ch-bio-004|ch-bio-006|cz-bio-001|'
    r'de-öko-\d+|es-eco-\d+|fr-bio-\d+|it-bio-\d+|nl-bio-01|pl-eko-\d+|pt-bio-\d+|'
    
    # Otros alfabetos (Cirílico/Griego/Asiático presentes en la lista)
    r'био|биологично|biological'
    r')\b'
)

# 2. Creamos la columna booleana
# 'case=False' ignora mayúsculas/minúsculas
# 'na=False' trata los valores nulos como False (no bio)
food_countries['is_bio'] = food_countries['labels'].str.contains(bio_pattern, case=False, na=False)

# 3. Verificamos el resultado
conteo = food_countries['is_bio'].value_counts()
porcentaje_bio = (food_countries['is_bio'].mean() * 100).round(2)

print(f"Resultados de la clasificación:")
print(conteo)
print(f"\nEl {porcentaje_bio}% de los productos han sido clasificados como Bio/Eco/Organic.")

Resultados de la clasificación:
is_bio
False    8641
True     1097
Name: count, dtype: int64

El 11.27% de los productos han sido clasificados como Bio/Eco/Organic.


## Extracción de Ingredientes Clave

In [9]:

"""
Primero creamos un archivo de texto con la lista completa de ingredientes del dataset, para después indetificar con IA generativa
todas las formas en las que aparecen los ingredientes clave y crear un diccionario para identificarlos
"""
# Unimos todo, lo pasamos a minúsculas y separamos por comas o espacios
todas_las_palabras = ",".join(food['ingredients_text'].fillna("").str.lower())
palabras_sueltas = pd.Series(todas_las_palabras.replace(';', ',').split(',')).str.strip().unique()

# Guardamos esta lista limpia de términos únicos
with open('terminos_unicos_ingredientes.txt', 'w', encoding='utf-8') as f:
    for palabra in sorted(palabras_sueltas):
        f.write(f"{palabra}\n")

In [10]:
"""
Con el texto generado, creamos un diccionario para crear una funcion que genere columnas booleanas que indican la 
presencia o ausencia de ingredientes clave en cada producto
"""

def extraer_ingredientes_clave(df, columna='ingredients_text'):
    df_ing = df.copy()
    
    # Diccionario maestro con Regex tamizado (Multilingüe)
    # \b asegura que buscamos palabras completas
    # el uso de (?:...) es para agrupar sin capturar, lo que lo hace más rápido
    dicc_ingredientes = {
        # GRASAS Y ACEITES
        'has_palm_oil': r'palm(?:a|e|iste|fat|fett|öl)|aceite.*palma|huile.*palme',
        'has_olive_oil': r'oliv(?:a|e|enöl)|azeite.*oliva|huile.*olive',
        'has_sunflower_oil': r'girasol|sunflower|tournesol|sonnenblumen',
        'has_rapeseed_oil': r'colza|rapeseed|canola|rapsöl|nabina',
        'has_coconut_oil': r'coco(?:nut|tier|s)?|koko(?:s|söl|sfett)',
        
        # CEREALES
        'has_wheat': r'trig(?:o|u)|wheat|blé|weizen|frumento',
        'has_oat': r'aven(?:a|e)|oat|avoine|hafer',
        'has_corn': r'ma(?:íz|iz|is|ize)|corn|korns?|granoturco',
        'has_rye': r'centeno|rye|seigle|roggen|segale',
        'has_barley': r'cebad(?:a)|barley|orge|gerst(?:e|en)',
        'has_rice': r'arroz|rice|riz|reiss?|riso',
        
        # AZÚCARES
        'has_added_sugar': r'sugar|sucre|zucchero|azúcar|zucker|açúcar|dextros(?:e|a)|fructos(?:e|a)|glucos(?:e|a)',
        'has_syrup': r'syrup|sirop|sciroppo|jarabe|xarope|sirup',
        
        # OTROS
        'has_soy': r'soja|soy(?:bean)?|soya',
        'has_nuts': r'hazelnut|noisette|avellan|mandeln?|almond|amand|anacard|cashew|walnut|noix'
    }
    
    print(">>> Extrayendo ingredientes clave mediante Regex...")
    for col, pattern in dicc_ingredientes.items():
        # Usamos regex=True explícitamente para mayor seguridad
        df_ing[col] = df_ing[columna].str.contains(pattern, case=False, na=False, regex=True)
        conteo = df_ing[col].sum()
        print(f"  - {col}: {conteo} productos detectados.")
        
    # Bonus: Contador de aditivos (Proxys de Grado NOVA)
    # Busca la letra E seguida de 3 o 4 números (formato estándar europeo)
    df_ing['additive_count'] = df_ing[columna].str.findall(r'E\d{3,4}', flags=re.IGNORECASE).str.len().fillna(0)
    
    return df_ing

# Aplicación
food_enriched = extraer_ingredientes_clave(food_countries)

>>> Extrayendo ingredientes clave mediante Regex...
  - has_palm_oil: 436 productos detectados.
  - has_olive_oil: 451 productos detectados.
  - has_sunflower_oil: 1971 productos detectados.
  - has_rapeseed_oil: 1605 productos detectados.
  - has_coconut_oil: 896 productos detectados.
  - has_wheat: 2492 productos detectados.
  - has_oat: 939 productos detectados.
  - has_corn: 815 productos detectados.
  - has_rye: 346 productos detectados.
  - has_barley: 859 productos detectados.
  - has_rice: 1071 productos detectados.
  - has_added_sugar: 4353 productos detectados.
  - has_syrup: 1054 productos detectados.
  - has_soy: 1934 productos detectados.
  - has_nuts: 890 productos detectados.


In [11]:
food_enriched.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9738 entries, 0 to 9803
Data columns (total 40 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                9738 non-null   int64  
 1   product_name        9504 non-null   object 
 2   brands              9364 non-null   object 
 3   countries           9738 non-null   object 
 4   quantity            8751 non-null   object 
 5   categories          9617 non-null   object 
 6   labels              7498 non-null   object 
 7   nutriscore_grade    9735 non-null   object 
 8   ecoscore_grade      9736 non-null   object 
 9   nova_group          8898 non-null   float64
 10  ingredients_text    9341 non-null   object 
 11  energy-kcal_100g    9215 non-null   float64
 12  fat_100g            9244 non-null   float64
 13  saturated-fat_100g  9077 non-null   float64
 14  carbohydrates_100g  9219 non-null   float64
 15  sugars_100g         9103 non-null   float64
 16  fiber_100g 

## Análisis e imputación de valores nulos

In [12]:
# Porcentaje de nulos por columna
null_report = (food_enriched.isnull().sum() / len(food_enriched) * 100).round(2)
print("Reporte de Nulos (%):")
print(null_report[null_report > 0].sort_values(ascending=False))

Reporte de Nulos (%):
fiber_100g            30.28
labels                23.00
quantity              10.14
nova_group             8.63
sodium_100g            6.79
saturated-fat_100g     6.79
salt_100g              6.79
sugars_100g            6.52
energy-kcal_100g       5.37
carbohydrates_100g     5.33
proteins_100g          5.12
fat_100g               5.07
ingredients_text       4.08
brands                 3.84
product_name           2.40
categories             1.24
nutriscore_grade       0.03
ecoscore_grade         0.02
dtype: float64


In [13]:
# Analizamos el % de nulos de fibra por cada una de tus categorías unificadas
fibra_por_cat = food_countries.groupby('category_unified')['fiber_100g'].apply(lambda x: x.isnull().mean() * 100).sort_values(ascending=False)

print("¿En qué categorías falta más la fibra? (%)")
print(fibra_por_cat)

¿En qué categorías falta más la fibra? (%)
category_unified
supplements_nutrition       88.888889
Otros                       87.050360
dairy_eggs                  54.181601
beverage                    48.191214
baby_food                   46.666667
meat_fish                   45.558739
fats_sauces                 42.962963
spices_baking_sugars        40.845070
alcoholic_beverages         40.000000
snacks_sweets               23.794832
plant_based_alternatives    17.523810
plant_based                 15.172414
ready_to_eat                 8.755274
cereals_bread                5.432937
Name: fiber_100g, dtype: float64


In [14]:
# ==============================================================================
# ESTRATEGIA DE IMPUTACIÓN CONDICIONAL PARA LA FIBRA (fiber_100g)
# ==============================================================================

# 1. Imputamos con 0 en categorías donde la fibra NO existe biológicamente
# Basado en el análisis previo de nulos por categoría.
cats_cero_biologico = [
    'dairy_eggs',           # Lácteos y huevos
    'meat_fish',            # Carnes y pescados
    'beverage',             # Bebidas líquidas (aguas, refrescos)
    'alcoholic_beverages',  # Bebidas alcohólicas
    'fats_sauces',          # Grasas y aceites (mayonesas, aceites)
    'supplements_nutrition' # Suplementos aislados (proteína whey, etc.)
]

# 2. Aplicamos la imputación solo a estas categorías específicas
# Esto evita que un pan o unas lentejas sin dato se conviertan en "0 fibra" por error.
mask_cero = food_enriched['category_unified'].isin(cats_cero_biologico)
food_enriched.loc[mask_cero, 'fiber_100g'] = food_enriched.loc[mask_cero, 'fiber_100g'].fillna(0)

# 3. Para las categorías de origen vegetal (Cereales, Plant-based, Snacks),
# eliminamos los nulos restantes. Si aquí falta la fibra, es un dato perdido (Missing)
# y no podemos arriesgarnos a sesgar las medias hacia abajo.
food_enriched = food_enriched.dropna(subset=['fiber_100g'])

# 4. Verificación final de integridad
nulos_finales = food_enriched['fiber_100g'].isna().sum()
print(f"Imputación completada. Nulos residuales en fibra: {nulos_finales}")
print(f"Tamaño actual del dataset: {len(food_enriched)} filas.")

Imputación completada. Nulos residuales en fibra: 0
Tamaño actual del dataset: 8652 filas.


In [15]:
# ==============================================================================
# GESTIÓN DEL RESTO DE NULOS
# ==============================================================================

# 1. FILTRADO POR INTEGRIDAD NUTRICIONAL (Eliminación de registros incompletos)
# Si faltan macronutrientes base o energía, el producto no es apto para un EDA riguroso.
cols_nutricionales_criticas = [
    'energy-kcal_100g', 'proteins_100g', 'fat_100g', 
    'saturated-fat_100g', 'carbohydrates_100g', 
    'sugars_100g', 'salt_100g'
]

# Creamos una copia limpia eliminando filas con nulos en estas columnas clave
food_enriched = food_enriched.dropna(subset=cols_nutricionales_criticas).copy()

# 2. TRATAMIENTO DE VARIABLES CATEGÓRICAS Y CALCULADAS
# Rellenamos nombres y marcas para evitar errores en agrupaciones y visualizaciones
food_enriched['product_name'] = food_enriched['product_name'].fillna('Producto sin nombre')
food_enriched['brands'] = food_enriched['brands'].fillna('Marca desconocida')

# Para NOVA, usamos 0 como valor centinela para "No Clasificado"
food_enriched['nova_group'] = food_enriched['nova_group'].fillna(0).astype(int)

# 3. REPORTE FINAL DE ESTADO
print(f"✅ Limpieza finalizada.")
print(f"📊 Tamaño final del dataset: {food_enriched.shape[0]} filas y {food_enriched.shape[1]} columnas.")
print(f"📉 Nulos residuales en nutrientes críticos: {food_enriched[cols_nutricionales_criticas].isnull().sum().sum()}")

✅ Limpieza finalizada.
📊 Tamaño final del dataset: 8030 filas y 40 columnas.
📉 Nulos residuales en nutrientes críticos: 0


In [16]:
# ==============================================================================
# LIMPIEZA FINAL DE ESTRUCTURA Y TEXTO DE INGREDIENTES
# ==============================================================================

# 1. Eliminamos la columna 'quantity' por baja relevancia analítica
if 'quantity' in food_enriched.columns:
    food_enriched = food_enriched.drop(columns=['quantity'])

# 2. Gestión de nulos en 'ingredients_text'
# No podemos borrar productos sin ingredientes (porque tenemos sus valores nutricionales),
# pero necesitamos que el texto sea procesable.
food_enriched['ingredients_text'] = food_enriched['ingredients_text'].fillna('Ingredientes no informados')

# 3. Gestión de nulos y normalización de 'labels' (Etiquetas)
# Crítico para que el motor de búsqueda de 'is_bio' y sellos de calidad no falle
food_enriched['labels'] = food_enriched['labels'].fillna('sin etiquetas')
food_enriched['labels'] = food_enriched['labels'].str.lower().str.strip()

# 4. Gestión de Categorías (Crítico para la integridad del mapeo)
# Rellenamos los 11 nulos para evitar errores en funciones de filtrado o búsqueda
food_enriched['categories'] = food_enriched['categories'].fillna('no clasificado')
food_enriched['categories'] = food_enriched['categories'].str.lower().str.strip()

# 5. Reseteamos el índice para que sea continuo y eliminamos el antiguo
food_enriched = food_enriched.reset_index(drop=True)

# REPORTE DE CIERRE
print("✅ Estructura final optimizada.")
print(f"Total registros listos para el EDA: {len(food_enriched)}")
print(f"Columnas disponibles: {list(food_enriched.columns)}")

✅ Estructura final optimizada.
Total registros listos para el EDA: 8030
Columnas disponibles: ['code', 'product_name', 'brands', 'countries', 'categories', 'labels', 'nutriscore_grade', 'ecoscore_grade', 'nova_group', 'ingredients_text', 'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'main_country', 'category_unified', 'corporation', 'is_bio', 'has_palm_oil', 'has_olive_oil', 'has_sunflower_oil', 'has_rapeseed_oil', 'has_coconut_oil', 'has_wheat', 'has_oat', 'has_corn', 'has_rye', 'has_barley', 'has_rice', 'has_added_sugar', 'has_syrup', 'has_soy', 'has_nuts', 'additive_count']


In [17]:
food_enriched.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8030 entries, 0 to 8029
Data columns (total 39 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                8030 non-null   int64  
 1   product_name        8030 non-null   object 
 2   brands              8030 non-null   object 
 3   countries           8030 non-null   object 
 4   categories          8030 non-null   object 
 5   labels              8030 non-null   object 
 6   nutriscore_grade    8030 non-null   object 
 7   ecoscore_grade      8030 non-null   object 
 8   nova_group          8030 non-null   int64  
 9   ingredients_text    8030 non-null   object 
 10  energy-kcal_100g    8030 non-null   float64
 11  fat_100g            8030 non-null   float64
 12  saturated-fat_100g  8030 non-null   float64
 13  carbohydrates_100g  8030 non-null   float64
 14  sugars_100g         8030 non-null   float64
 15  fiber_100g          8030 non-null   float64
 16  protei

## Ya tenemos el dataset procesado y listo para exportar

In [19]:
# Guardar el CSV procesado
path_relativo = '../data/processed/foods_cleaned.csv'
food_enriched.to_csv(path_relativo, index=False)